In [1]:
from generate_knowledge_graph_qwen3_base import *
import os
import numpy as np
import json

from tqdm import tqdm
import asyncio
from tqdm.asyncio import tqdm_asyncio
from openai import AsyncOpenAI
from openai import OpenAI
import pandas as pd
import math
import time
import requests
from lxml import html
from lxml import etree
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [2]:
PROJECT_ROOT = Path("../../").resolve()

In [3]:
with open(PROJECT_ROOT / 'data' / 'aid_assay_name_map.json', 'r', encoding='utf-8') as f:
    loaded_map = json.load(f)

In [ ]:
uri = "bolt://localhost:7687"
userName = "neo4j"
password = "your_neo4j_password"
openrouter_base_url="https://openrouter.ai/api/v1"
openrouter_api_key="your_openrouter_api_key"

EMBEDDING_MODEL="text-embedding-v4"
toxicity="hepatotoxicity"

In [ ]:
models = ["openai/gpt-5.2","google/gemini-3-pro-preview", "deepseek/deepseek-v3.2","z-ai/glm-4.7","qwen/qwen3-235b-a22b-2507"]

In [ ]:
graph = create_graph_database_connection(uri, userName, password)

In [ ]:
EMBEDDING_FUNCTION, dimension = load_embedding_model(EMBEDDING_MODEL)

In [ ]:

folder_path = f"./{toxicity}_data/{toxicity}_AOP_Abstract"
rel_assay_path = f"./{toxicity}_data/{toxicity}_AOP_rel_assay"


folder_path.mkdir(parents=True, exist_ok=True)
rel_assay_path.mkdir(parents=True, exist_ok=True)

In [ ]:
def get_toxicity_aop_by_search(toxicity):
    base_url = f"https://aopwiki.org/aops?utf8=%E2%9C%93&search={toxicity}"
    response = requests.get(base_url, verify=False)
    response.encoding = "utf-8"
    tree = html.fromstring(response.content)
    aop_list = tree.xpath('//*[@id="aop-fulltext-table"]/tbody/tr/td[1]/text()')
    aop_list.extend(tree.xpath('//*[@id="aop-index-table"]/tbody/tr/td[1]/text()'))

    aop_list = list(set(aop_list))
    print(f"Direct search found {len(aop_list)} AOPs for {toxicity}")
    return aop_list

In [ ]:
def get_toxicity_aops_url_by_events(toxicity):

    url = f"https://aopwiki.org/events?search={toxicity}"

    response = requests.get(url,  verify=False)
    response.raise_for_status()
    parser = etree.HTMLParser()
    tree = etree.fromstring(response.text, parser)
    
    base_url = "https://aopwiki.org"
    xpath_query = "//h3[@id='fulltext-search']/following-sibling::div[1]//table[@id='event-index-table']//tbody//tr//td[2]//a/@href"
    
    relative_links = tree.xpath(xpath_query)
    event_full_urls = [base_url + link for link in relative_links]
    print("Key Events Fulltext Search Results:", len(event_full_urls))
    return event_full_urls

In [ ]:
def rel_events_to_aops(event_url ):
    print("Processing Event URL:", event_url)
    response = requests.get(event_url, verify=False)
    response.raise_for_status()
    
    parser = etree.HTMLParser()
    tree = etree.fromstring(response.text, parser)
    
    row_xpath = '//*[@id="aops"]/div[2]/div/table/tbody/tr'
    rows = tree.xpath(row_xpath)
    
    aop_id = []
    
    for row in rows:
        link = row.xpath('./td[1]/a/@href')
        if link:
            full_url =  link[0].split('/aops/')[-1] 
            aop_id.append(full_url)
    
    print(f"Found {len(aop_id)} AOPs for Event URL: {event_url}")
    return aop_id

In [ ]:
def get_toxicity_aop_by_events( toxicity):
    event_full_urls = get_toxicity_aops_url_by_events(toxicity)
    aop_id_by_event = []
    for event_url in event_full_urls:
        aop_id_by_event.extend(rel_events_to_aops(event_url ))
    # Deduplicate
    aop_id_by_event=list(set(aop_id_by_event))
    print(f"Event-based search found {len(aop_id_by_event)} AOPs for {toxicity}")
    return aop_id_by_event

In [ ]:
aop_id_by_event = get_toxicity_aop_by_events( toxicity)

In [ ]:
aop_id_by_search = get_toxicity_aop_by_search(toxicity)

In [ ]:
all_aop_ids = list(set(aop_id_by_event + aop_id_by_search))

In [ ]:
len(all_aop_ids)

In [ ]:
def download_aops_abstract(aop_list,folder_path):
    count = 0
    for aop in aop_list:
        aop_base_url = f"https://aopwiki.org/aops/{aop}"
        
        response = requests.get(aop_base_url, verify=False)
        response.encoding = "utf-8"
        tree = html.fromstring(response.content)

        title_list = tree.xpath('//*[@id="aop_title"]/div[2]/h1/text()')
        aop_title = title_list[0].strip() 

        abstract_text_list = tree.xpath('//*[@id="abstract"]/*[@class ="section-content"]//text()')


        if not abstract_text_list or len("".join(abstract_text_list).strip()) == 0:
            print(f"AOP {aop} has no abstract content")
            continue

        abstract_text_str = "".join(abstract_text_list).strip()
        abstract_text_str = " ".join(abstract_text_str.split()) # Remove \xa0 and extra spaces

        file_name = f"{folder_path}/AOP_{aop}_abstract.txt"
        with open(file_name, "w", encoding="utf-8") as f:
            f.write(f"Title: {aop_title}\n")
            f.write("-" * 20 + "\n")
            f.write(f"Abstract:\n{abstract_text_str}")
        
        count += 1
        print(f"Downloaded successfully: {aop} - {aop_title}")
        time.sleep(0.5)  

In [ ]:
download_aops_abstract(aop_list=all_aop_ids, folder_path=folder_path)

In [ ]:
AOP_Abstract = os.listdir(folder_path)

In [ ]:
folder_path

In [ ]:
len(AOP_Abstract)

In [ ]:
aop_id_list = []
aop_abstract_list = []
for aop_file in AOP_Abstract:
    aop_id = aop_file.split("_")[1]
    aop_id_list.append(aop_id)

    with open(f"{folder_path}/{aop_file}", "r") as f:
        abstract = f.read()
    aop_abstract_list.append(abstract)

In [ ]:
df_aop_abstract = pd.DataFrame({"aop_id": aop_id_list, "abstract": aop_abstract_list})

In [ ]:
df_aop_abstract.head()

In [ ]:
conn = Neo4jConnection(uri,userName, password)

In [ ]:
query_content = df_aop_abstract.loc[1, "abstract"]
docs = get_relevant_assay(query_content, graph, EMBEDDING_FUNCTION)
doc, sources, chunkdetails = format_documents(docs)
directly_similar_assays = [i.split("_")[0] for i in sources]
chunk_id_df = pd.DataFrame([i[0] for i in chunkdetails])
chunk_id_unique = chunk_id_df.drop_duplicates(subset=["id"])
content_rel_assay = pd.DataFrame()
for i in chunk_id_unique.index:
    chunk_id = chunk_id_unique.loc[i, "id"]
    chunk_rel_assay_df = get_chunk_rel_aids( conn=conn,chunk_id=chunk_id, conn_counts=10)
    chunk_rel_assay_df = pd.DataFrame(
        {
            "assay_id": [i.split("_")[0] for i in chunk_rel_assay_df.index.values],
            "count": chunk_rel_assay_df.values,
        }
    )
    content_rel_assay = pd.concat([content_rel_assay, chunk_rel_assay_df], axis=0)

In [ ]:
chunk_id_df


In [ ]:
chunk_id_unique

In [ ]:
content_rel_assay.groupby("assay_id", as_index=False)[
        "count"
    ].sum()["assay_id"]

In [ ]:
len(directly_similar_assays)

In [ ]:
df_aop_abstract

In [ ]:
def aop2rel_assay(query_content):
    docs = get_relevant_assay(query_content, graph, EMBEDDING_FUNCTION)
    doc, sources, chunkdetails = format_documents(docs)
    directly_similar_assays = [i.split("_")[0] for i in sources]
    chunk_id_df = pd.DataFrame([i[0] for i in chunkdetails])
    chunk_id_unique = chunk_id_df.drop_duplicates(subset=["id"])
    content_rel_assay = pd.DataFrame()
    for i in chunk_id_unique.index:
        chunk_id = chunk_id_unique.loc[i, "id"]
        chunk_rel_assay_df = get_chunk_rel_aids(conn=conn, chunk_id=chunk_id, conn_counts=10)
        chunk_rel_assay_df = pd.DataFrame(
            {
                "assay_id": [i.split("_")[0] for i in chunk_rel_assay_df.index.values],
                "count": chunk_rel_assay_df.values,
            }
        )
        content_rel_assay = pd.concat([content_rel_assay, chunk_rel_assay_df], axis=0)
    content_rel_assay = content_rel_assay.reset_index(drop=True)
    indirectly_related_assays = content_rel_assay.groupby("assay_id", as_index=False)[
        "count"
    ].sum()
    indirectly_related_assays_sorted = indirectly_related_assays.sort_values(
        by="count", ascending=False
    )
    for i in indirectly_related_assays_sorted.index:
        if (
            indirectly_related_assays_sorted.loc[i, "assay_id"]
            in directly_similar_assays
        ):
            indirectly_related_assays_sorted.drop(i, inplace=True)
    directly_similar = pd.DataFrame({"assay_id": directly_similar_assays})
    directly_similar["related_rank"] = 1
    indirectly_related_assays_sorted["related_rank"] = 2
    aop_similar_assay = pd.concat(
        [directly_similar, indirectly_related_assays_sorted], axis=0
    )
    aop_similar_assay.reset_index(drop=True, inplace=True)

    for i in aop_similar_assay.index:
        assay_id = aop_similar_assay.loc[i, "assay_id"]
        with open(f"PROJECT_ROOT / data / bioassays_description/ aid_test_100/{assay_id}_description.txt", "r") as f:
            assay_description = f.read()
        assay_name= loaded_map.get(str(assay_id))
    
        assay_name_description= f"**BioAssay Name**:{assay_name}\n**BioAssay Description**:{assay_description}"
        
        aop_similar_assay.loc[i, "assay_description"] = assay_name_description

    return aop_similar_assay

In [ ]:
def rerank_assay_by_qwen3(query_content,aop_similar_assay):
    resp = dashscope.TextReRank.call(
        model="qwen3-rerank",
        query=query_content,
        documents=[
            passage for passage in aop_similar_assay.loc[:, "assay_description"].values
        ],
        top_n=aop_similar_assay.shape[0],
        return_documents=False,
        instruct="Given a scientific description of a biological mechanism or adverse outcome pathway, retrieve relevant bioassay descriptions, experimental protocols, or data resources used to evaluate these biological effects."

    )
    
    rerank_result_df = pd.DataFrame(resp.output.results).loc[:,["index","relevance_score"]]

    rerank_result_df = rerank_result_df.sort_values(by="index")


    aop_similar_assay = aop_similar_assay.copy() # Avoid SettingWithCopyWarning
    aop_similar_assay["relevance_score"] = rerank_result_df["relevance_score"].values
    
    
    return    aop_similar_assay

In [ ]:
df_all_aop_similar_assay = pd.DataFrame()

for i in df_aop_abstract.index:
    print(f"Processing item {i}, AOP: {df_aop_abstract.loc[i,'aop_id']}")
    query_content = df_aop_abstract.loc[i, "abstract"]
    aop_similar_assay = aop2rel_assay(query_content)
    aop_similar_assay["aop_source"] = df_aop_abstract.loc[i, "aop_id"]
    aop_similar_assay["aop_abstract"] = query_content
    df_all_aop_similar_assay = pd.concat(
        [df_all_aop_similar_assay, aop_similar_assay], axis=0
    )
    time.sleep(0.1)
    

In [ ]:
df_all_aop_similar_assay

In [ ]:
df_all_aop_similar_assay.reset_index(drop=True, inplace=True)

In [ ]:
len(df_all_aop_similar_assay["aop_abstract"].unique())

In [ ]:
f"{rel_assay_path}/{toxicity}_aop_similar_assay_qwen3.csv"

In [ ]:
df_all_aop_similar_assay.to_csv(f"{rel_assay_path}/{toxicity}_aop_similar_assay_qwen3.csv", index=False,encoding='utf-8-sig')

In [ ]:
df_all_aop_similar_assay= pd.read_csv(f"{rel_assay_path}/{toxicity}_aop_similar_assay_qwen3.csv",encoding='utf-8-sig')

In [ ]:
result = df_all_aop_similar_assay.groupby('aop_source')['assay_id'].count()
print(result)

In [ ]:
API_BATCH_LIMIT = 50 
TARGET_TOP_N = 50

def call_qwen_rerank(query, documents, top_n):
    if not documents:
        return []

    if len(documents) > API_BATCH_LIMIT:
        print(f"    [Warning] Input length {len(documents)} exceeds limit {API_BATCH_LIMIT}. Truncating.")
        documents = documents[:API_BATCH_LIMIT]
        
    actual_top_n = min(top_n, len(documents))
    
    try:
        resp = dashscope.TextReRank.call(
            model="qwen3-rerank",
            query=query,
            documents=documents,
            top_n=actual_top_n,
            return_documents=False,
            instruct="Given a scientific description of a biological mechanism, retrieve relevant bioassay descriptions."
        )
        
        if resp.status_code == HTTPStatus.OK:
            return resp.output.results
        else:
            print(f"    [API Error] {resp.code}: {resp.message}")
            return []
    except Exception as e:
        print(f"    [Exception] {e}")
        return []

def get_sorted_df_from_results(original_df, results):
    """
    Helper function: build a sorted DataFrame from API results
    """
    if not results:
        return pd.DataFrame()
    
    sorted_rows = []

    for res in results:
        if res.index < len(original_df):
            row = original_df.iloc[res.index].copy()
            row['relevance_score'] = res.relevance_score
            sorted_rows.append(row)
            
    return pd.DataFrame(sorted_rows)

def process_tournament_sort(query, df_group):

    total_count = len(df_group)
    num_batches = math.ceil(total_count / API_BATCH_LIMIT)
    sub_dfs = np.array_split(df_group, num_batches)
    keep_per_batch = math.ceil(TARGET_TOP_N / num_batches)
    
    survivors = []
    for i, sub_df in enumerate(sub_dfs):
        batch_docs = sub_df['assay_description'].tolist()
        current_top_n = min(keep_per_batch, len(batch_docs))
        results = call_qwen_rerank(query, batch_docs, top_n=current_top_n)
        batch_survivors = get_sorted_df_from_results(sub_df, results)
        survivors.append(batch_survivors)
        time.sleep(0.1)
    if not survivors:
        return pd.DataFrame()
    
    candidates_df = pd.concat(survivors, ignore_index=True)
    if len(candidates_df) > API_BATCH_LIMIT:
        candidates_df = candidates_df.sort_values(by='relevance_score', ascending=False).head(API_BATCH_LIMIT)
    
    final_docs = candidates_df['assay_description'].tolist()
    final_results = call_qwen_rerank(query, final_docs, top_n=TARGET_TOP_N)
    final_df = get_sorted_df_from_results(candidates_df, final_results)
    
    return final_df

def rerank_assay_strict_limit(df_all):
    print("Starting Strict-Limit Rerank Process...")
    
    df_all = df_all.copy()
    df_all['assay_description'] = df_all['assay_description'].fillna("-").astype(str)
    
    grouped = df_all.groupby('aop_source')
    final_results = []
    
    for i, (aop_id, group) in enumerate(grouped):
        count = len(group)
        
        if count <= TARGET_TOP_N:

            g = group.copy()
            g['relevance_score'] = np.nan
            final_results.append(g)
            
        else:
            print(f"Group {i} (Size {count}): Tournament Sort")
            
            query = group['aop_abstract'].iloc[0]
            top_50_df = process_tournament_sort(query, group)
            final_results.append(top_50_df)
            
    if final_results:
        return pd.concat(final_results, ignore_index=True)
    else:
        return pd.DataFrame(columns=df_all.columns)

In [ ]:
df_all_aop_similar_assay_rerank = rerank_assay_strict_limit(df_all_aop_similar_assay)

In [ ]:
df_all_aop_similar_assay_rerank.to_csv(f"{rel_assay_path}/{toxicity}_aop_similar_assay_rerank_qwen3.csv", index=False,encoding='utf-8-sig')

In [ ]:
result = df_all_aop_similar_assay_rerank.groupby('aop_source')['assay_id'].count()
print(result)

In [ ]:
df_all_aop_similar_assay_rerank

In [ ]:
df_all_aop_similar_assay_rerank

In [ ]:
client = AsyncOpenAI(
  base_url=openrouter_base_url,
  api_key=openrouter_api_key,
)

async def process_single_assay(sem, index, row, model, toxicity):

    aop_description = row["aop_abstract"]
    assay_description = row["assay_description"]

    if row["relevance_score"] < 0.0: 
        return index, None, None

    async with sem:  
        max_retries = 3
        for attempt in range(max_retries):
            try:
                response = await client.chat.completions.create(
                    model=model,
        messages=[
            {
                "role": "system",

                "content": f"You are an expert toxicologist. You need to determine whether the positive substances in the bio-assay have potential {toxicity} based on the provided AOP knowledge. You must answer in a valid JSON format.",
            },
            {
                "role": "user",
                "content": f"""
Please analyze the following information:

    AOP Information:
    ----
    {aop_description}
    ----

    Bio-assay Description:
    ----
    {assay_description}
    ----

    Task:
    Determine whether the positive substance in the bio-assay has a high potential {toxicity} risk.

    Output Instructions:
    Please provide your response as a JSON object with the following keys:
    1. "reasoning": A detailed step-by-step analysis connecting the assay's endpoint to the AOP's Key Events. Explain why you think there is or isn't a link.
    2. "final_answer": Return string "T" if there is a potential {toxicity} risk, or "F" if not.
    """,
            },
        ],
            temperature=0.1,
            response_format={"type": "json_object"}, 
            

                )
                result_json = json.loads(response.choices[0].message.content)
                if isinstance(result_json["reasoning"], list):
                    reasoning_content = "\n".join(result_json["reasoning"])
                else:
                    reasoning_content = result_json["reasoning"]
                final_answer = result_json["final_answer"]
                
                return index, final_answer, reasoning_content

            except Exception as e:
                if attempt < max_retries - 1:
                    await asyncio.sleep(1 + attempt) 
                else:
                    print(f"Index {index} failed after {max_retries} attempts: {e}")
                    return index, "Error", f"Error: {str(e)}"

async def main_concurrent(df, model, toxicity, max_concurrency=10):
    sem = asyncio.Semaphore(max_concurrency) 
    tasks = []
    for i, row in df.iterrows():
        task = process_single_assay(sem, i, row, model, toxicity)
        tasks.append(task)

    results = await tqdm_asyncio.gather(*tasks, desc=f"Processing ({model})")
    
    return results

In [ ]:
models = ["openai/gpt-5.2","google/gemini-3-pro-preview", "deepseek/deepseek-v3.2","z-ai/glm-4.7","qwen/qwen3-235b-a22b-2507"]
for target_model in models[1:]:
    target_toxicity = toxicity 
    CONCURRENCY_LIMIT = 10
    print(f"Starting concurrent processing with concurrency limit: {CONCURRENCY_LIMIT}...")

    final_results = await main_concurrent(df_all_aop_similar_assay_rerank, target_model, target_toxicity, CONCURRENCY_LIMIT)

    for res in final_results:
        if res is None: continue 
        
        idx, content, reasoning = res
        
        if content is not None: 
            df_all_aop_similar_assay_rerank.loc[idx, f"Potential_{target_toxicity} ({target_model})"] = content
            df_all_aop_similar_assay_rerank.loc[idx, f"LLM Reasoning ({target_model})"] = reasoning

    print("Processing completed!")

In [ ]:
df_all_aop_similar_assay_rerank

In [ ]:
df_all_aop_similar_assay_rerank[df_all_aop_similar_assay_rerank[f"Potential_{target_toxicity} ({target_model})"]=="T"]

In [ ]:
models = ["openai/gpt-5.2","google/gemini-3-pro-preview", "deepseek/deepseek-v3.2","z-ai/glm-4.7","qwen/qwen3-235b-a22b-2507"]

In [ ]:

for target_model in models:
    target_toxicity = toxicity 
    CONCURRENCY_LIMIT = 10
    print(f"Starting concurrent processing with concurrency limit: {CONCURRENCY_LIMIT}...")
    final_results = await main_concurrent(df_all_aop_similar_assay_rerank[df_all_aop_similar_assay_rerank[f"Potential_{target_toxicity} ({target_model})"]=="Error"], target_model, target_toxicity, CONCURRENCY_LIMIT)

    for res in final_results:
        if res is None: continue 
        
        idx, content, reasoning = res
        
        if content is not None: 
            df_all_aop_similar_assay_rerank.loc[idx, f"Potential_{target_toxicity} ({target_model})"] = content
            df_all_aop_similar_assay_rerank.loc[idx, f"LLM Reasoning ({target_model})"] = reasoning

    print("Processing completed!")

In [ ]:

columns_to_check = [
    f"Potential_{target_toxicity} ({models[0]})",
    f"Potential_{target_toxicity} ({models[1]})",
    f"Potential_{target_toxicity} ({models[2]})",
    f"Potential_{target_toxicity} ({models[3]})",
    f"Potential_{target_toxicity} ({models[4]})"
]
df_all_aop_similar_assay_rerank['T_score'] = (df_all_aop_similar_assay_rerank[columns_to_check] == "T").sum(axis=1)

In [ ]:
all_model_T = df_all_aop_similar_assay_rerank[df_all_aop_similar_assay_rerank['T_score'] >=5]

In [ ]:
all_model_T["assay_id"].nunique()

In [ ]:
all_model_T

In [ ]:
df_all_aop_similar_assay_rerank.to_csv(f"./{toxicity}_data/{toxicity}_aop_similar_assay_LLMs_reasoning.csv",index=None,encoding="utf-8-sig")

In [ ]:
df_all_aop_similar_assay_rerank=pd.read_csv(f"./{toxicity}_data/{toxicity}_aop_similar_assay_LLMs_reasoning.csv",encoding="utf-8-sig")

In [ ]:
df_all_aop_similar_assay_rerank["assay_id_aop_id"]=df_all_aop_similar_assay_rerank["assay_id"].astype(str)+"_"+df_all_aop_similar_assay_rerank["aop_source"].astype(str)    

In [ ]:
df_all_aop_similar_assay_rerank

In [ ]:
venn_diagram_data = pd.DataFrame()

for model in models:     
    df_drop = df_all_aop_similar_assay_rerank[
        df_all_aop_similar_assay_rerank[f"Potential_{toxicity} ({model})"] == "T"
    ].sort_values(by="relevance_score", ascending=False).drop_duplicates(subset=["assay_id_aop_id"], keep="first")
    model_assay_id = df_drop["assay_id_aop_id"]
    model_assay_id.name = model
    model_assay_id = model_assay_id.reset_index(drop=True)
    
    print(f"{model}: {model_assay_id.shape[0]}")

    venn_diagram_data = pd.concat([venn_diagram_data, model_assay_id], axis=1)
venn_diagram_data.to_csv(f"{toxicity}_data/{toxicity}_aop_similar_assay_rerank_qwen3_venn_data.csv", index=False,encoding='utf-8-sig')


In [ ]:
folder_path

In [ ]:
df_all_aop_similar_assay_rerank.to_csv(f"./{toxicity}_data/aop_similar_assay_rerank_llm_infer.csv",index=None,encoding="utf-8-sig")

In [ ]:
df_last = df_all_aop_similar_assay_rerank

In [ ]:
start = 0
end = 0.9
step = 0.01

start_int = int(start / step)
end_int = int(end / step)

decimal_numbers = [i * step for i in range(start_int, end_int + 1)]

In [ ]:
cut_t_rate = []
for cut in decimal_numbers:
    df_cut = df_last[df_last["relevance_score"] >= cut]
    rate = np.sum(df_cut[f"Potential_{toxicity} ({model})"] == "T") / df_cut.shape[0]
    cut_t_rate.append(rate)

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["SimHei"]  
plt.rcParams["axes.unicode_minus"] = False  
plt.title("Proportion of T predictions across rerank_score thresholds")
plt.xlabel("rerank_score")
plt.ylabel("Proportion predicted as T")
plt.plot(decimal_numbers, cut_t_rate, marker="o")

#### Download CSV

In [ ]:
df_all_aop_similar_assay_rerank = pd.read_csv(f"./{toxicity}_data/{toxicity}_aop_similar_assay_LLMs_reasoning.csv",encoding="utf-8-sig")

In [ ]:
columns_to_check = [
    f"Potential_{toxicity} ({models[0]})",
    f"Potential_{toxicity} ({models[1]})",
    f"Potential_{toxicity} ({models[2]})",
    f"Potential_{toxicity} ({models[3]})",
    f"Potential_{toxicity} ({models[4]})"
]

In [ ]:
df_all_aop_similar_assay_rerank['T_score'] = (df_all_aop_similar_assay_rerank[columns_to_check] == "T").sum(axis=1)

In [ ]:
df_all_aop_similar_assay_rerank[df_all_aop_similar_assay_rerank['T_score'] ==5]

In [ ]:
assay_need_down=df_all_aop_similar_assay_rerank[df_all_aop_similar_assay_rerank['T_score'] ==5]

In [ ]:
assay_need_down

In [ ]:
assay_need_down.to_csv(f"./{toxicity}_data/aop_similar_assay_rerank_llm_infer_T.csv",encoding="utf-8-sig",index=None)

In [ ]:
len(assay_need_down.assay_id.to_list())

In [ ]:
assay_need_down.drop_duplicates(subset=["assay_id"],keep="first",inplace=True)

In [ ]:
len(assay_need_down.assay_id.to_list())

In [ ]:
from get_info_pub import Get_Data_Pub

assay_loader = Get_Data_Pub(
    aid_list=assay_need_down.assay_id.to_list(), task_name=f"{toxicity}"
)
assay_loader.get_aid_csv()